Obtain ID's of records that have zero subjects.  
Then free to experiment with methods to enrich their metadata.

In [19]:
NO_RECORDS = 150

In [20]:
from datetime import datetime, timezone
from json import JSONDecodeError
from logging.config import dictConfig
from typing import Any, Optional
import psycopg
from psycopg import errors as psycopg_errors
from psycopg.rows import dict_row
from config.logging_config import LOGGING_CONFIG
from config.postgres_config import PostgresConfig
from tasks import transform_batch, add_file_metadata
import os
from fastapi import FastAPI, Query, HTTPException
import logging
from pydantic import BaseModel, Field
from utils.queue_utils import HarvestEventQueue, detect_identifier_type

from lxml import etree as ET
# API calls
import requests


# added for these experiments
from utils.enrichment_methods import get_doi, enrich_subjects_from_openaire

In [21]:
dictConfig(LOGGING_CONFIG)
logger = logging.getLogger(__name__)

In [22]:
user = os.getenv("POSTGRES_USER")
passwd = os.getenv("POSTGRES_PASSWORD")
host = os.getenv("POSTGRES_ADDRESS", "127.0.0.1")

if host == '':
    host = "127.0.0.1"

connection_params = {
    "user" : user,
    "password" : passwd,
    "dbname" : "datasetdb",
    "port" : 5432,
    "host" : host
}

In [23]:
query_records = """
            -- Exploring subjects from record raw metadata

-- Every xpath defines a potential XML structure with corresponding subject tag placement.
-- If raw_metadata conforms to ANY of these xpaths, subjects will be concatenated <subject> tags. (removing array_to_string() will return arrays)
-- If raw_metadata doesn't conform to any path (and probably has no subject tags), subjects will be ''


WITH query AS (
    SELECT
        records.*,
        array_to_string(
                xpath('oai:record/oai:metadata/datacite:resource/datacite:subjects/datacite:subject/text()', raw_metadata,
                    '{{datacite, http://datacite.org/schema/kernel-4},
                      {oai, http://www.openarchives.org/OAI/2.0/}}'
                )::text[] ||
				xpath('oai:record/oai:metadata/oaidc:oai_datacite/oaidc:payload/datacite:resource/datacite:subjects/datacite:subject/text()', raw_metadata,
                    '{{datacite, http://datacite.org/schema/kernel-4},
                      {oai, http://www.openarchives.org/OAI/2.0/},
                      {oaidc, http://schema.datacite.org/oai/oai-1.1/}}'
                )::text[] ||
				xpath('oai:record/oai:metadata/oai:resource/datacite:subjects/datacite:subject/text()', raw_metadata,
                    '{{datacite, http://datacite.org/schema/kernel-4},
                      {oai, http://www.openarchives.org/OAI/2.0/}}'
                )::text[] ||
				xpath('oai:record/oai:metadata/oad:oai_datacite/oad:payload/datacite:resource/datacite:subjects/datacite:subject/text()', raw_metadata,
				-- note the 3 in datacite
                    '{{datacite, http://datacite.org/schema/kernel-3},
					  {oai, http://www.openarchives.org/OAI/2.0/},
     				  {oad, http://schema.datacite.org/oai/oai-1.0/}}'
                )::text[] ||
				-- same path as previous but uses kernel-4 namespace
				xpath('oai:record/oai:metadata/oad:oai_datacite/oad:payload/datacite:resource/datacite:subjects/datacite:subject/text()', raw_metadata,
                    '{{datacite, http://datacite.org/schema/kernel-4},
					  {oai, http://www.openarchives.org/OAI/2.0/},
     				  {oad, http://schema.datacite.org/oai/oai-1.0/}}'
                )::text[]
				, ', '
        ) AS subjects
    FROM public.records
)
SELECT 
    id,
    title,
    raw_metadata,
    subjects
FROM query
WHERE query.subjects = ''
LIMIT %s
;
"""

In [24]:
try:
    with psycopg.connect(**connection_params, row_factory=dict_row) as conn:
        cur = conn.cursor()
        
        cur.execute(query_records, (NO_RECORDS,))
        
        results = cur.fetchall()
        print(results)

        record_ids = [str(row["id"]) for row in results]
        record_titles = [str(row["title"]) for row in  results]
        record_raw_metadata = [str(row["raw_metadata"]) for row in  results]


except Exception as e:
    logger.exception("Failed to fetch harvest runs")
    raise HTTPException(status_code=500, detail=str(e))

[{'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/1001518064', 'title': 'Inhouse research', 'raw_metadata': '<record xmlns="http://www.openarchives.org/OAI/2.0/" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance">\n  <header>\n    <identifier>oai:icatplus.esrf.fr:inv/1001518064</identifier>\n    <datestamp>2022-11-18T07:00:15.457Z</datestamp>\n  </header>\n  <metadata>\n    <oai_datacite xmlns="http://schema.datacite.org/oai/oai-1.1/" xsi:schemaLocation="http://schema.datacite.org/oai/oai-1.1/ http://schema.datacite.org/oai/oai-1.1/oai.xsd">\n      <schemaVersion>4.2</schemaVersion>\n      <datacentreSymbol>EI</datacentreSymbol>\n      <payload>\n        <resource xmlns="http://schema.datacite.org/meta/kernel-4.2/" xsi:schemaLocation="http://schema.datacite.org/meta/kernel-4.2/ http://schema.datacite.org/meta/kernel-4.2/metadata.xsd"><identifier identifierType="DOI">10.15151/ESRF-ES-1001518064</identifier><creators><creator><creatorName nameType="Personal">William CHEVREMONT</creat

In [25]:
record_titles

['Inhouse research',
 'X-RAY CRYSTALLOGRAPHY AND SAXS STUDIES OF PROTEINS RELATED TO PATHOGENESIS OF TRANSMISSIBLE AND NON-TRANSMISSIBLE DISEASES, CELL DIVISION, S',
 'CHRONOS',
 'CryoEM-BAG France: Structural Biology using cryoEM in France',
 'Effects of high hydrostatic pressure on crystallographic structure of xynalase 11 to characterize the conformations of the thumb-loop involv',
 'Temperature study of Magnetic signatures in photo-switchable thin crystals',
 'Kmapping of Ge microdisks',
 'Structural Biology BAG - Portugal',
 'Mechanisms of Human Disease',
 'XRF mapping on perovskite thin layers',
 'Cryo-electron microscopy EPN BAG',
 'Pressure-induced valence changes as origin of magnetism loss in Ce2Pd2In',
 'Operando XAS analysis of visible light assisted CO2 reduction on mono- and bi-metallic (Ni, Cu,Fe)/ CeO2 catalysts',
 '3D orientation and strain mapping during hydrogen-based direct reduction of iron oxides',
 'Operando investigations on ternary In2O3-based catalysts for met

In [26]:
record_raw_metadata[2]

'<record xmlns="http://www.openarchives.org/OAI/2.0/" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance">\n  <header>\n    <identifier>oai:icatplus.esrf.fr:inv/2113708027</identifier>\n    <datestamp>2025-06-05T06:00:10.495Z</datestamp>\n  </header>\n  <metadata>\n    <oai_datacite xmlns="http://schema.datacite.org/oai/oai-1.1/" xsi:schemaLocation="http://schema.datacite.org/oai/oai-1.1/ http://schema.datacite.org/oai/oai-1.1/oai.xsd">\n      <schemaVersion>4.2</schemaVersion>\n      <datacentreSymbol>EI</datacentreSymbol>\n      <payload>\n        <resource xmlns="http://schema.datacite.org/meta/kernel-4.2/" xsi:schemaLocation="http://schema.datacite.org/meta/kernel-4.2/ http://schema.datacite.org/meta/kernel-4.2/metadata.xsd"><identifier identifierType="DOI">10.15151/ESRF-ES-2113708027</identifier><creators><creator><creatorName nameType="Personal">Bag CHRONOS</creatorName><givenName>Bag</givenName><familyName>Chronos</familyName></creator><creator><creatorName nameType="Personal"

In [27]:
get_doi(record_raw_metadata[2])

'10.15151/ESRF-ES-2113708027'

# DOI check through bigger repositories  (#1)
Take an XML ETree.
Searches for identifier with identifierType DOI.
    - what if there are multiple, and one is under "related works"?
Seaches thru OpenAlex or Openaire - apis?
Data available? N -> end
Y ->
    search for fields: subject, topic, field, discipline, keyword

OpenAIRE API
- they're officialy moving to a Graph API in May 2026
https://api.openaire.eu/graph/swagger-ui/index.html

In [28]:
import logging
logging.getLogger("urllib3").setLevel(logging.WARNING)

In [29]:
doi = get_doi(record_raw_metadata[0]) 

if doi == []:
    print("NO DOI")
else:
    print(enrich_subjects_from_openaire(doi))

['PaN', 'IH-LS-3430', 'ID02', 'Particles, Nuclei and Fields', 'In-house Life Science', 'IH-LS-3434']


In [30]:
# get only records that have a DOI
ix_has_doi = []

for i in range(0, len(record_raw_metadata)):
    rec = get_doi(record_raw_metadata[i])
    if doi != []:
        ix_has_doi.append(i)

print(ix_has_doi)

[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149]


In [31]:
id_to_enrich = [record_ids[i] for i in ix_has_doi]
doi_to_enrich = [get_doi(record_raw_metadata[i]) for i in ix_has_doi]

print(id_to_enrich)
print(doi_to_enrich)

['5d1a3aed-114c-43da-b80b-a953b9424daa::inv/1001518064', '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/2323561777', '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/2113708027', '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/1776323595', '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/649170553', '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/1497239752', '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/90632078', '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/2233771865', '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/1934195855', '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/1205127635', '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/1299132129', '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/1427751371', '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/644969439', '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/894233253', '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/1081709123', '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/1446521728', '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/1329162612', '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/701214588

In [32]:
# limit to smaller section
#id_to_enrich = id_to_enrich[:30]
#doi_to_enrich = doi_to_enrich[:30]

In [33]:
from utils.enrichment_methods import _openaire_batch_doi_resolve, _opeanaire_get_doi_subjects, _ids_connect_with_openaire_subjects, batch_enrich_subjects_from_openaire

api_result = _openaire_batch_doi_resolve(doi_to_enrich)

print(api_result[:3])

[{'authors': [{'fullName': 'Canciani Anselmo, Anselmo', 'name': 'Anselmo', 'surname': 'Canciani Anselmo', 'rank': 1, 'pid': {'id': {'scheme': 'orcid_pending', 'value': '0000-0001-6390-3459'}, 'provenance': None}}, {'fullName': 'Di Pisa, Flavio', 'name': 'Flavio', 'surname': 'Di Pisa', 'rank': 2, 'pid': {'id': {'scheme': 'orcid_pending', 'value': '0000-0003-2302-1464'}, 'provenance': None}}, {'fullName': 'Forneris, Federico', 'name': 'Federico', 'surname': 'Forneris', 'rank': 3, 'pid': {'id': {'scheme': 'orcid_pending', 'value': '0000-0002-7818-1804'}, 'provenance': None}}, {'fullName': 'Marabelli, Chiara', 'name': 'Chiara', 'surname': 'Marabelli', 'rank': 4, 'pid': None}, {'fullName': 'Ricci, Caterina', 'name': 'Caterina', 'surname': 'Ricci', 'rank': 5, 'pid': {'id': {'scheme': 'orcid_pending', 'value': '0000-0003-0573-4065'}, 'provenance': None}}], 'openAccessColor': None, 'publiclyFunded': None, 'type': 'dataset', 'language': {'code': 'eng', 'label': 'English'}, 'countries': None, 's

In [34]:
doi_subjects = _opeanaire_get_doi_subjects(api_result)
doi_subjects

[{'doi': '10.15151/esrf-es-2323561777',
  'subjects': ['SAXS, Small-Angle X-ray Scattering',
   'BM29',
   'MX-2790',
   'Macromolecular Crystallography']},
 {'doi': '10.15151/esrf-es-1497239752',
  'subjects': ['IH-HC-3980',
   'ID09',
   'In-house Hard Condensed Matter Science']},
 {'doi': '10.15151/esrf-es-90632078',
  'subjects': ['IH-HC-3376',
   'id01',
   'In-house Hard Condensed Matter Science']},
 {'doi': '10.15151/esrf-es-1205127635',
  'subjects': ['ID13', 'IH-MA-368', '7. Clean energy']},
 {'doi': '10.15151/esrf-es-2233771865',
  'subjects': ['BM29', 'Macromolecular Crystallography', 'MX-2736']},
 {'doi': '10.15151/esrf-es-2112610773',
  'subjects': ['PaN',
   'Earth Sciences',
   'x-ray probe',
   'Particles, Nuclei and Fields',
   'BM05',
   'BM18',
   'ES-1503']},
 {'doi': '10.15151/esrf-es-2113174295',
  'subjects': ['PaN',
   'Earth Sciences',
   'x-ray probe',
   'Particles, Nuclei and Fields',
   'BM05',
   'BM18',
   'ES-1503']},
 {'doi': '10.15151/esrf-es-211317436

In [35]:
id_dois = []
for ix in range(len(id_to_enrich)):
    id_dois.append(
        {
            'id': id_to_enrich[ix],
            'doi': doi_to_enrich[ix].lower(),
        }
    )
id_dois

[{'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/1001518064',
  'doi': '10.15151/esrf-es-1001518064'},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/2323561777',
  'doi': '10.15151/esrf-es-2323561777'},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/2113708027',
  'doi': '10.15151/esrf-es-2113708027'},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/1776323595',
  'doi': '10.15151/esrf-es-1776323595'},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/649170553',
  'doi': '10.15151/esrf-es-649170553'},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/1497239752',
  'doi': '10.15151/esrf-es-1497239752'},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/90632078',
  'doi': '10.15151/esrf-es-90632078'},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/2233771865',
  'doi': '10.15151/esrf-es-2233771865'},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/1934195855',
  'doi': '10.15151/esrf-es-1934195855'},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/1205127635

In [36]:
enriched_list = _ids_connect_with_openaire_subjects(id_dois, doi_subjects)
enriched_list

[{'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/1001518064',
  'doi': '10.15151/esrf-es-1001518064',
  'enriched_subjects': {'ID02',
   'IH-LS-3430',
   'IH-LS-3434',
   'In-house Life Science',
   'PaN',
   'Particles, Nuclei and Fields'}},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/2323561777',
  'doi': '10.15151/esrf-es-2323561777',
  'enriched_subjects': {'BM29',
   'MX-2790',
   'Macromolecular Crystallography',
   'SAXS, Small-Angle X-ray Scattering'}},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/2113708027',
  'doi': '10.15151/esrf-es-2113708027',
  'enriched_subjects': {'BM05',
   'BM18',
   'ES-1503',
   'Earth Sciences',
   'PaN',
   'Particles, Nuclei and Fields',
   'x-ray probe'}},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/1776323595',
  'doi': '10.15151/esrf-es-1776323595',
  'enriched_subjects': {'CM01',
   'MX-2596',
   'MX-2704',
   'Macromolecular Crystallography',
   'PaN',
   'Particles, Nuclei and Fields',
   'electron microscopy'}},
 {'i

In [37]:
# Only succesfully enriched IDs and subjects.
# Could be useful for comparing how many ids we sent to be enriched vs
# how many actually got new subjects
[el for el in enriched_list if el['enriched_subjects'] is not None]

[{'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/1001518064',
  'doi': '10.15151/esrf-es-1001518064',
  'enriched_subjects': {'ID02',
   'IH-LS-3430',
   'IH-LS-3434',
   'In-house Life Science',
   'PaN',
   'Particles, Nuclei and Fields'}},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/2323561777',
  'doi': '10.15151/esrf-es-2323561777',
  'enriched_subjects': {'BM29',
   'MX-2790',
   'Macromolecular Crystallography',
   'SAXS, Small-Angle X-ray Scattering'}},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/2113708027',
  'doi': '10.15151/esrf-es-2113708027',
  'enriched_subjects': {'BM05',
   'BM18',
   'ES-1503',
   'Earth Sciences',
   'PaN',
   'Particles, Nuclei and Fields',
   'x-ray probe'}},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/1776323595',
  'doi': '10.15151/esrf-es-1776323595',
  'enriched_subjects': {'CM01',
   'MX-2596',
   'MX-2704',
   'Macromolecular Crystallography',
   'PaN',
   'Particles, Nuclei and Fields',
   'electron microscopy'}},
 {'i

In [38]:
# All-in-one function

batch_enrich_subjects_from_openaire(id_dois)

[{'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/1001518064',
  'doi': '10.15151/esrf-es-1001518064',
  'enriched_subjects': {'ID02',
   'IH-LS-3430',
   'IH-LS-3434',
   'In-house Life Science',
   'PaN',
   'Particles, Nuclei and Fields'}},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/2323561777',
  'doi': '10.15151/esrf-es-2323561777',
  'enriched_subjects': {'BM29',
   'MX-2790',
   'Macromolecular Crystallography',
   'SAXS, Small-Angle X-ray Scattering'}},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/2113708027',
  'doi': '10.15151/esrf-es-2113708027',
  'enriched_subjects': {'BM05',
   'BM18',
   'ES-1503',
   'Earth Sciences',
   'PaN',
   'Particles, Nuclei and Fields',
   'x-ray probe'}},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/1776323595',
  'doi': '10.15151/esrf-es-1776323595',
  'enriched_subjects': {'CM01',
   'MX-2596',
   'MX-2704',
   'Macromolecular Crystallography',
   'PaN',
   'Particles, Nuclei and Fields',
   'electron microscopy'}},
 {'i

In [39]:
# Comparison: Number of initial records vs number of records that have gained subjects via this method
enriched = batch_enrich_subjects_from_openaire(id_dois)
print(len([el['doi'] for el in enriched]))
print(len([el['doi'] for el in enriched if el['enriched_subjects'] is not None]))

150
150


In [40]:
# Taking a look at the metadata initially obtained from DB, subset of few records
full_info_enriched = [el for el in enriched if el['enriched_subjects'] is not None]

first_enriched = full_info_enriched[:10]

In [41]:
# indexes that have been enriched
ix_original = [record_ids.index(rec['id']) for rec in first_enriched]
ix_original

[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]

In [42]:
first_enriched

[{'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/1001518064',
  'doi': '10.15151/esrf-es-1001518064',
  'enriched_subjects': {'ID02',
   'IH-LS-3430',
   'IH-LS-3434',
   'In-house Life Science',
   'PaN',
   'Particles, Nuclei and Fields'}},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/2323561777',
  'doi': '10.15151/esrf-es-2323561777',
  'enriched_subjects': {'BM29',
   'MX-2790',
   'Macromolecular Crystallography',
   'SAXS, Small-Angle X-ray Scattering'}},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/2113708027',
  'doi': '10.15151/esrf-es-2113708027',
  'enriched_subjects': {'BM05',
   'BM18',
   'ES-1503',
   'Earth Sciences',
   'PaN',
   'Particles, Nuclei and Fields',
   'x-ray probe'}},
 {'id': '5d1a3aed-114c-43da-b80b-a953b9424daa::inv/1776323595',
  'doi': '10.15151/esrf-es-1776323595',
  'enriched_subjects': {'CM01',
   'MX-2596',
   'MX-2704',
   'Macromolecular Crystallography',
   'PaN',
   'Particles, Nuclei and Fields',
   'electron microscopy'}},
 {'i

In [43]:
[record_raw_metadata[i] for i in ix_original]

['<record xmlns="http://www.openarchives.org/OAI/2.0/" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance">\n  <header>\n    <identifier>oai:icatplus.esrf.fr:inv/1001518064</identifier>\n    <datestamp>2022-11-18T07:00:15.457Z</datestamp>\n  </header>\n  <metadata>\n    <oai_datacite xmlns="http://schema.datacite.org/oai/oai-1.1/" xsi:schemaLocation="http://schema.datacite.org/oai/oai-1.1/ http://schema.datacite.org/oai/oai-1.1/oai.xsd">\n      <schemaVersion>4.2</schemaVersion>\n      <datacentreSymbol>EI</datacentreSymbol>\n      <payload>\n        <resource xmlns="http://schema.datacite.org/meta/kernel-4.2/" xsi:schemaLocation="http://schema.datacite.org/meta/kernel-4.2/ http://schema.datacite.org/meta/kernel-4.2/metadata.xsd"><identifier identifierType="DOI">10.15151/ESRF-ES-1001518064</identifier><creators><creator><creatorName nameType="Personal">William CHEVREMONT</creatorName><givenName>William</givenName><familyName>Chevremont</familyName><nameIdentifier nameIdentifierScheme

## OpenAlex API
https://developers.openalex.org/
https://developers.openalex.org/guides/authentication

Get multiple works by DOI

use | between DOIs, up to 100 DOIs at once!!
check out how the results look then.

https://api.openalex.org/works?filter=doi:https://doi.org/10.1234/a|https://doi.org/10.1234/b|https://doi.org/10.1234/c&per_page=100



Use the select parameter to return only specific fields:
https://api.openalex.org/works?select=id,doi,display_name

In [44]:
OpenAlex_API_key = os.getenv("OPENALEX_API_KEY")

In [45]:
OPENALEX_API_URL = "https://api.openalex.org/works/"

doi = '10.15151/esrf-es-1473312981'

# We have to append the ID directly in the path
query_url = OPENALEX_API_URL + "https://doi.org/" + doi

response = requests.get(
    query_url,
    params={"api_key": OpenAlex_API_key}, 
    timeout=30
)

data = response.json()


In [46]:
data

{'id': 'https://openalex.org/W6886229830',
 'doi': 'https://doi.org/10.15151/esrf-es-1473312981',
 'title': 'SAXS WAXS measurements on reference samples for different sample detector distances. Control of transmission measurements',
 'display_name': 'SAXS WAXS measurements on reference samples for different sample detector distances. Control of transmission measurements',
 'publication_year': 2024,
 'publication_date': '2024-01-25',
 'ids': {'openalex': 'https://openalex.org/W6886229830',
  'doi': 'https://doi.org/10.15151/esrf-es-1473312981'},
 'language': 'en',
 'primary_location': {'id': 'doi:10.15151/esrf-es-1473312981',
  'is_oa': True,
  'landing_page_url': 'https://doi.org/10.15151/esrf-es-1473312981',
  'pdf_url': None,
  'source': {'id': 'https://openalex.org/S7407051764',
   'display_name': 'European Synchrotron Radiation Facility',
   'issn_l': None,
   'issn': [],
   'is_oa': False,
   'is_in_doaj': False,
   'is_core': False,
   'host_organization': None,
   'host_organiza

In [64]:
OPENALEX_API_URL = "https://api.openalex.org/works"

def fetch_works_by_dois(dois, api_key):
    all_results = []
    chunk_size = 100

    # Strip any existing https://doi.org/ or http://doi.org/ prefix
    # so we can add it back consistently for every DOI
    clean_dois = [
        doi.replace("https://doi.org/", "")
           .replace("http://doi.org/", "")
        for doi in dois
    ]

    for i in range(0, len(clean_dois), chunk_size):
        chunk = clean_dois[i : i + chunk_size]

        # Join with pipe — each DOI gets the full https://doi.org/ prefix
        doi_filter = "|".join(f"https://doi.org/{doi}" for doi in chunk)

        # Build URL as f-string to avoid requests encoding the colons and pipes
        url = f"{OPENALEX_API_URL}?filter=doi:{doi_filter}&per_page={chunk_size}&api_key={api_key}"

        response = requests.get(url, timeout=30)
        response.raise_for_status()
        all_results.extend(response.json().get("results", []))

    return all_results

In [65]:
fetch_works_by_dois(['10.15151/esrf-es-1001518064',
                     '10.15151/esrf-es-2323561777'],
                    api_key=OpenAlex_API_key)

[{'id': 'https://openalex.org/W7154628459',
  'doi': 'https://doi.org/10.15151/esrf-es-2323561777',
  'title': 'X-RAY CRYSTALLOGRAPHY AND SAXS STUDIES OF PROTEINS RELATED TO PATHOGENESIS OF TRANSMISSIBLE AND NON-TRANSMISSIBLE DISEASES, CELL DIVISION, S',
  'display_name': 'X-RAY CRYSTALLOGRAPHY AND SAXS STUDIES OF PROTEINS RELATED TO PATHOGENESIS OF TRANSMISSIBLE AND NON-TRANSMISSIBLE DISEASES, CELL DIVISION, S',
  'publication_year': 2026,
  'publication_date': '2026-04-15',
  'ids': {'openalex': 'https://openalex.org/W7154628459',
   'doi': 'https://doi.org/10.15151/esrf-es-2323561777'},
  'language': 'en',
  'primary_location': {'id': 'doi:10.15151/esrf-es-2323561777',
   'is_oa': True,
   'landing_page_url': 'https://doi.org/10.15151/esrf-es-2323561777',
   'pdf_url': None,
   'source': {'id': 'https://openalex.org/S7407051764',
    'display_name': 'European Synchrotron Radiation Facility',
    'issn_l': None,
    'issn': [],
    'is_oa': False,
    'is_in_doaj': False,
    'is_cor

In [ ]:
# Get potential subjects, two relevant fields: keywords and concepts
# Concepts is an older system, keywords is newer - but concepts is more diverse

# DOI resolver method  (#2)
Take an XML ETree.
Searches for identifier with identifierType DOI.
    - what if there are multiple, and one is under "related works"?
Talks to an API such as DataCite or SignPosting.
Data available? N -> end
Y ->
    search for fields: subject, topic, field, discipline, keyword